# Section 0: Configuration and Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os
import json
from tqdm import tqdm
from IPython.display import display

plt.style.use('ggplot')
np.random.seed(42)

CONFIG = {
    'data_dir': r'data',
    'results_dir': r'results',
    'src_dir': r'src',
    'random_state': 42
}
os.makedirs(CONFIG['results_dir'], exist_ok=True)


# Section 1: Introduction and Research Question
**Research Question**: Can machine learning models trained on structured medical abstracts reliably classify them by research type (Diagnosis, Treatment, Prevention)?


# Section 2: Data Loading and Leakage Prevention


In [ ]:
import sys
sys.path.append('.')
from src.preprocess import load_data, reconstruct_abstracts, map_labels, balance_dataset

df_lines = load_data()
df_abs = reconstruct_abstracts(df_lines)
df_labeled = map_labels(df_abs) # This now strips keywords to prevent leakage
df_balanced = balance_dataset(df_labeled, target_size=20000)
display(df_balanced.head())


# Section 4: Visualisations


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, label in enumerate(['Treatment', 'Diagnosis', 'Prevention']):
    text = ' '.join(df_balanced[df_balanced['label'] == label]['abstract_text'].sample(500))
    wc = WordCloud(width=400, height=400, background_color='white').generate(text)
    axes[i].imshow(wc)
    axes[i].set_title(label)
    axes[i].axis('off')
plt.savefig(os.path.join(CONFIG['results_dir'], 'wordclouds.png'))
plt.show()


# Section 5: Correct Train/Test Split and Pipeline Models


In [ ]:
from sklearn.preprocessing import LabelEncoder
from src.baseline_model import load_split_data, train_logreg, train_nb
from src.evaluate import evaluate_model

X_train, X_test, y_train, y_test = load_split_data()
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
classes = le.classes_.tolist()

print('Data split complete. Train size:', len(X_train))


# Section 6: Model 1 — Logistic Regression (Pipeline)


In [ ]:
mod1 = train_logreg(X_train, y_train_enc, le)
y_pred1 = le.inverse_transform(mod1.predict(X_test))
y_prob1 = mod1.predict_proba(X_test)
metrics1 = evaluate_model(y_test, y_pred1, y_prob1, classes, 'Logistic Regression', CONFIG['results_dir'])


# Section 7: Model 2 — Multinomial NB (Pipeline)


In [ ]:
mod2 = train_nb(X_train, y_train_enc, le)
y_pred2 = le.inverse_transform(mod2.predict(X_test))
y_prob2 = mod2.predict_proba(X_test)
metrics2 = evaluate_model(y_test, y_pred2, y_prob2, classes, 'Multinomial NB', CONFIG['results_dir'])


# Section 9: Model Comparison Table


In [ ]:
res_df = pd.DataFrame([metrics1, metrics2])
display(res_df)
